# 🔮 Trading Forecasting - Inferência dos Modelos

Este notebook implementa a inferência usando os modelos treinados para prever descontos aplicáveis.

## Objetivos:
- Carregar modelos treinados
- Realizar previsões para novos dados
- Avaliar incertezas das previsões
- Gerar relatórios de previsão
- Simular cenários de negociação

In [ ]:
# Importações necessárias
import pandas as pd
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurações
plt.style.use('seaborn-v0_8')
np.random.seed(42)

## 1. Carregamento dos Modelos

In [ ]:
def load_trained_models():
    """Carrega todos os modelos treinados"""
    models_dir = "../models"
    loaded_models = {}
    
    if not os.path.exists(models_dir):
        print("❌ Diretório de modelos não encontrado. Execute primeiro o notebook 02_model_training.ipynb")
        return {}
    
    # Listar arquivos de modelo
    model_files = [f for f in os.listdir(models_dir) if f.startswith('modelo_material_') and f.endswith('.joblib')]
    
    print(f"📁 Encontrados {len(model_files)} arquivos de modelo")
    
    for model_file in model_files:
        # Extrair código do material do nome do arquivo
        codigo_material = model_file.replace('modelo_material_', '').replace('.joblib', '')
        
        try:
            # Carregar modelo
            model_path = os.path.join(models_dir, model_file)
            model = joblib.load(model_path)
            loaded_models[codigo_material] = model
            print(f"✅ Modelo carregado para material {codigo_material}")
        except Exception as e:
            print(f"❌ Erro ao carregar modelo {model_file}: {str(e)}")
    
    return loaded_models

# Carregar modelos
models = load_trained_models()
print(f"\n🎯 Total de modelos carregados: {len(models)}")
print(f"Materiais disponíveis: {list(models.keys())}")

In [ ]:
# Carregar métricas de treinamento se disponíveis
try:
    with open('../models/metrics/training_metrics.json', 'r') as f:
        training_metrics = json.load(f)
    print("✅ Métricas de treinamento carregadas")
    
    # Mostrar resumo das métricas
    print("\n📊 Resumo das métricas dos modelos:")
    for material, metrics in training_metrics.items():
        print(f"   Material {material}: MAE = {metrics['mae_test']:.2f}%, R² = {metrics['r2_test']:.3f}")
except FileNotFoundError:
    training_metrics = {}
    print("⚠️ Métricas de treinamento não encontradas")

## 2. Função de Previsão

In [ ]:
def predict_discount(codigo_material, fornecedor, quantidade, prazo_entrega, 
                    condicoes_pagamento, frete_incluso, validade_proposta, preco_unitario):
    """Realiza previsão de desconto para um material específico"""
    
    # Verificar se modelo existe para o material
    if codigo_material not in models:
        available_materials = list(models.keys())
        return {
            'erro': f'Modelo não encontrado para material {codigo_material}',
            'materiais_disponiveis': available_materials
        }
    
    # Preparar dados de entrada
    input_data = pd.DataFrame({
        'Fornecedor': [fornecedor],
        'Quantidade': [quantidade],
        'PrazoEntrega(dias)': [prazo_entrega],
        'CondicoesPagamento': [condicoes_pagamento],
        'FreteIncluso': [frete_incluso],
        'ValidadeProposta(dias)': [validade_proposta],
        'PrecoUnitario(R$)': [preco_unitario]
    })
    
    # Fazer previsão
    modelo = models[codigo_material]
    desconto_previsto = modelo.predict(input_data)[0]
    
    # Calcular intervalo de confiança usando estimadores do Random Forest
    rf_regressor = modelo.named_steps['regressor']
    
    # Previsões de cada árvore individual
    tree_predictions = []
    for estimator in rf_regressor.estimators_:
        # Transformar dados usando o preprocessador
        X_transformed = modelo.named_steps['preprocessor'].transform(input_data)
        pred = estimator.predict(X_transformed)[0]
        tree_predictions.append(pred)
    
    # Estatísticas das previsões
    std_prediction = np.std(tree_predictions)
    lower_bound = desconto_previsto - 1.96 * std_prediction
    upper_bound = desconto_previsto + 1.96 * std_prediction
    
    # Métricas do modelo (se disponíveis)
    model_metrics = training_metrics.get(codigo_material, {})
    
    resultado = {
        'codigo_material': codigo_material,
        'desconto_previsto': round(float(desconto_previsto), 2),
        'intervalo_confianca': {
            'limite_inferior': round(float(lower_bound), 2),
            'limite_superior': round(float(upper_bound), 2)
        },
        'incerteza_std': round(float(std_prediction), 2),
        'dados_entrada': {
            'fornecedor': fornecedor,
            'quantidade': quantidade,
            'prazo_entrega': prazo_entrega,
            'condicoes_pagamento': condicoes_pagamento,
            'frete_incluso': frete_incluso,
            'validade_proposta': validade_proposta,
            'preco_unitario': preco_unitario
        },
        'modelo_performance': {
            'mae_teste': model_metrics.get('mae_test', 'N/A'),
            'r2_teste': model_metrics.get('r2_test', 'N/A')
        },
        'timestamp': datetime.now().isoformat()
    }
    
    return resultado

print("✅ Função de previsão definida")

## 3. Exemplos de Previsão

In [ ]:
# Exemplo 1: Previsão básica
if models:
    # Usar primeiro material disponível
    material_exemplo = list(models.keys())[0]
    
    print(f"🧪 Exemplo de previsão para material {material_exemplo}:")
    
    resultado = predict_discount(
        codigo_material=material_exemplo,
        fornecedor='Alpha Ltda',
        quantidade=200,
        prazo_entrega=15,
        condicoes_pagamento='À vista',
        frete_incluso='Não',
        validade_proposta=30,
        preco_unitario=350.00
    )
    
    print(f"\n📈 Resultado da previsão:")
    print(f"   Desconto previsto: {resultado['desconto_previsto']}%")
    print(f"   Intervalo de confiança: {resultado['intervalo_confianca']['limite_inferior']}% - {resultado['intervalo_confianca']['limite_superior']}%")
    print(f"   Incerteza (std): ±{resultado['incerteza_std']}%")
    
    if resultado['modelo_performance']['mae_teste'] != 'N/A':
        print(f"   MAE do modelo: {resultado['modelo_performance']['mae_teste']:.2f}%")
else:
    print("❌ Nenhum modelo disponível para exemplo")

## 4. Análise de Sensibilidade

In [ ]:
# Análise de como diferentes variáveis afetam o desconto
def sensitivity_analysis(codigo_material, base_params):
    """Analisa sensibilidade do modelo a mudanças nos parâmetros"""
    
    if codigo_material not in models:
        print(f"❌ Modelo não disponível para material {codigo_material}")
        return
    
    print(f"🔍 Análise de sensibilidade para material {codigo_material}")
    
    # Parâmetros base
    base_prediction = predict_discount(codigo_material, **base_params)
    base_discount = base_prediction['desconto_previsto']
    
    print(f"\n📊 Desconto base: {base_discount}%")
    print(f"Parâmetros base: {base_params}")
    
    # Analisar variação de quantidade
    print(f"\n🔢 Impacto da QUANTIDADE:")
    quantidades = [100, 200, 300, 500, 1000]
    for qtd in quantidades:
        params = base_params.copy()
        params['quantidade'] = qtd
        pred = predict_discount(codigo_material, **params)
        desconto = pred['desconto_previsto']
        variacao = desconto - base_discount
        print(f"   Qtd {qtd:4d}: {desconto:6.2f}% (Δ{variacao:+.2f}%)")
    
    # Analisar variação de preço
    print(f"\n💰 Impacto do PREÇO UNITÁRIO:")
    precos = [200, 300, 400, 500, 600]
    for preco in precos:
        params = base_params.copy()
        params['preco_unitario'] = preco
        pred = predict_discount(codigo_material, **params)
        desconto = pred['desconto_previsto']
        variacao = desconto - base_discount
        print(f"   R$ {preco:3d}: {desconto:6.2f}% (Δ{variacao:+.2f}%)")
    
    # Analisar variação de prazo
    print(f"\n⏰ Impacto do PRAZO DE ENTREGA:")
    prazos = [5, 10, 15, 20, 30]
    for prazo in prazos:
        params = base_params.copy()
        params['prazo_entrega'] = prazo
        pred = predict_discount(codigo_material, **params)
        desconto = pred['desconto_previsto']
        variacao = desconto - base_discount
        print(f"   {prazo:2d} dias: {desconto:6.2f}% (Δ{variacao:+.2f}%)")

# Executar análise se houver modelos
if models:
    material_exemplo = list(models.keys())[0]
    
    params_base = {
        'fornecedor': 'Alpha Ltda',
        'quantidade': 200,
        'prazo_entrega': 15,
        'condicoes_pagamento': 'À vista',
        'frete_incluso': 'Não',
        'validade_proposta': 30,
        'preco_unitario': 350.00
    }
    
    sensitivity_analysis(material_exemplo, params_base)

## 5. Simulação de Cenários

In [ ]:
# Simular diferentes cenários de negociação
def simulate_scenarios(codigo_material):
    """Simula diferentes cenários de negociação"""
    
    if codigo_material not in models:
        print(f"❌ Modelo não disponível para material {codigo_material}")
        return
    
    print(f"🎭 Simulação de cenários para material {codigo_material}")
    
    # Definir cenários
    cenarios = {
        'Cenário Conservador': {
            'fornecedor': 'Alpha Ltda',
            'quantidade': 100,
            'prazo_entrega': 20,
            'condicoes_pagamento': '60 dias',
            'frete_incluso': 'Sim',
            'validade_proposta': 15,
            'preco_unitario': 400.00
        },
        'Cenário Padrão': {
            'fornecedor': 'Beta Corp',
            'quantidade': 200,
            'prazo_entrega': 15,
            'condicoes_pagamento': '30 dias',
            'frete_incluso': 'Não',
            'validade_proposta': 30,
            'preco_unitario': 350.00
        },
        'Cenário Agressivo': {
            'fornecedor': 'Gamma SA',
            'quantidade': 500,
            'prazo_entrega': 10,
            'condicoes_pagamento': 'À vista',
            'frete_incluso': 'Não',
            'validade_proposta': 45,
            'preco_unitario': 300.00
        }
    }
    
    resultados_cenarios = []
    
    for nome_cenario, params in cenarios.items():
        resultado = predict_discount(codigo_material, **params)
        resultados_cenarios.append((nome_cenario, resultado))
        
        print(f"\n📋 {nome_cenario}:")
        print(f"   Desconto previsto: {resultado['desconto_previsto']}%")
        print(f"   Intervalo: {resultado['intervalo_confianca']['limite_inferior']}% - {resultado['intervalo_confianca']['limite_superior']}%")
        print(f"   Configuração: {params['quantidade']} unid, {params['prazo_entrega']} dias, {params['condicoes_pagamento']}")
    
    # Visualização dos cenários
    nomes = [r[0] for r in resultados_cenarios]
    descontos = [r[1]['desconto_previsto'] for r in resultados_cenarios]
    lower_bounds = [r[1]['intervalo_confianca']['limite_inferior'] for r in resultados_cenarios]
    upper_bounds = [r[1]['intervalo_confianca']['limite_superior'] for r in resultados_cenarios]
    
    plt.figure(figsize=(12, 6))
    x_pos = range(len(nomes))
    
    # Barras principais
    bars = plt.bar(x_pos, descontos, alpha=0.7, capsize=5)
    
    # Barras de erro (intervalo de confiança)
    errors = [[d - l for d, l in zip(descontos, lower_bounds)],
              [u - d for u, d in zip(upper_bounds, descontos)]]
    plt.errorbar(x_pos, descontos, yerr=errors, fmt='none', capsize=5, color='black')
    
    plt.xlabel('Cenários')
    plt.ylabel('Desconto Previsto (%)')
    plt.title(f'Comparação de Cenários - Material {codigo_material}')
    plt.xticks(x_pos, nomes, rotation=45)
    
    # Adicionar valores nas barras
    for i, (bar, desconto) in enumerate(zip(bars, descontos)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{desconto:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return resultados_cenarios

# Executar simulação
if models:
    material_exemplo = list(models.keys())[0]
    cenarios_resultado = simulate_scenarios(material_exemplo)

## 6. Comparação Entre Fornecedores

In [ ]:
# Comparar previsões para diferentes fornecedores
def compare_suppliers(codigo_material, base_params, suppliers_list):
    """Compara previsões entre diferentes fornecedores"""
    
    if codigo_material not in models:
        print(f"❌ Modelo não disponível para material {codigo_material}")
        return
    
    print(f"🏢 Comparação de fornecedores para material {codigo_material}")
    
    resultados_fornecedores = []
    
    for fornecedor in suppliers_list:
        params = base_params.copy()
        params['fornecedor'] = fornecedor
        
        resultado = predict_discount(codigo_material, **params)
        resultados_fornecedores.append((fornecedor, resultado))
        
        print(f"\n📊 {fornecedor}:")
        print(f"   Desconto previsto: {resultado['desconto_previsto']}%")
        print(f"   Intervalo de confiança: {resultado['intervalo_confianca']['limite_inferior']}% - {resultado['intervalo_confianca']['limite_superior']}%")
    
    # Visualização
    fornecedores = [r[0] for r in resultados_fornecedores]
    descontos = [r[1]['desconto_previsto'] for r in resultados_fornecedores]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(fornecedores, descontos, alpha=0.7)
    plt.xlabel('Fornecedores')
    plt.ylabel('Desconto Previsto (%)')
    plt.title(f'Comparação de Fornecedores - Material {codigo_material}')
    plt.xticks(rotation=45)
    
    # Adicionar valores
    for bar, desconto in zip(bars, descontos):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{desconto:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Ranking
    ranking = sorted(resultados_fornecedores, key=lambda x: x[1]['desconto_previsto'], reverse=True)
    print(f"\n🏆 Ranking de fornecedores (maior desconto):")
    for i, (fornecedor, resultado) in enumerate(ranking, 1):
        print(f"   {i}º {fornecedor}: {resultado['desconto_previsto']}%")
    
    return resultados_fornecedores

# Executar comparação
if models:
    material_exemplo = list(models.keys())[0]
    
    params_comparacao = {
        'quantidade': 250,
        'prazo_entrega': 15,
        'condicoes_pagamento': 'À vista',
        'frete_incluso': 'Não',
        'validade_proposta': 30,
        'preco_unitario': 350.00
    }
    
    fornecedores_teste = ['Alpha Ltda', 'Beta Corp', 'Gamma SA', 'Delta Inc']
    
    comparacao_resultado = compare_suppliers(material_exemplo, params_comparacao, fornecedores_teste)

## 7. Relatório de Previsão Detalhado

In [ ]:
def generate_prediction_report(codigo_material, prediction_params):
    """Gera relatório detalhado de previsão"""
    
    print(f"📋 RELATÓRIO DE PREVISÃO - MATERIAL {codigo_material}")
    print("=" * 60)
    
    # Fazer previsão principal
    resultado = predict_discount(codigo_material, **prediction_params)
    
    if 'erro' in resultado:
        print(f"❌ {resultado['erro']}")
        return
    
    # Seção 1: Dados de Entrada
    print(f"\n📝 DADOS DA PROPOSTA:")
    dados = resultado['dados_entrada']
    print(f"   Fornecedor: {dados['fornecedor']}")
    print(f"   Quantidade: {dados['quantidade']:,} unidades")
    print(f"   Preço Unitário: R$ {dados['preco_unitario']:.2f}")
    print(f"   Prazo de Entrega: {dados['prazo_entrega']} dias")
    print(f"   Condições de Pagamento: {dados['condicoes_pagamento']}")
    print(f"   Frete Incluso: {dados['frete_incluso']}")
    print(f"   Validade da Proposta: {dados['validade_proposta']} dias")
    
    # Seção 2: Previsão
    print(f"\n🎯 PREVISÃO DE DESCONTO:")
    print(f"   Desconto Sugerido: {resultado['desconto_previsto']}%")
    print(f"   Intervalo de Confiança (95%): {resultado['intervalo_confianca']['limite_inferior']}% - {resultado['intervalo_confianca']['limite_superior']}%")
    print(f"   Incerteza (desvio padrão): ±{resultado['incerteza_std']}%")
    
    # Seção 3: Interpretação
    desconto = resultado['desconto_previsto']
    incerteza = resultado['incerteza_std']
    
    print(f"\n💡 INTERPRETAÇÃO:")
    if incerteza < 2:
        confiabilidade = "Alta"
    elif incerteza < 5:
        confiabilidade = "Média"
    else:
        confiabilidade = "Baixa"
    
    print(f"   Confiabilidade da Previsão: {confiabilidade}")
    
    if desconto < 5:
        categoria = "Baixo"
        recomendacao = "Condições pouco favoráveis para desconto"
    elif desconto < 15:
        categoria = "Moderado"
        recomendacao = "Desconto razoável esperado"
    else:
        categoria = "Alto"
        recomendacao = "Excelente oportunidade de desconto"
    
    print(f"   Categoria do Desconto: {categoria}")
    print(f"   Recomendação: {recomendacao}")
    
    # Seção 4: Performance do Modelo
    performance = resultado['modelo_performance']
    print(f"\n📊 PERFORMANCE DO MODELO:")
    if performance['mae_teste'] != 'N/A':
        print(f"   Erro Médio Absoluto: {performance['mae_teste']:.2f}%")
        print(f"   R² (Coeficiente de Determinação): {performance['r2_teste']:.3f}")
    else:
        print(f"   Métricas não disponíveis")
    
    # Seção 5: Valor Total
    valor_total = dados['quantidade'] * dados['preco_unitario']
    economia_estimada = valor_total * (desconto / 100)
    
    print(f"\n💰 IMPACTO FINANCEIRO:")
    print(f"   Valor Total da Proposta: R$ {valor_total:,.2f}")
    print(f"   Economia Estimada: R$ {economia_estimada:,.2f}")
    print(f"   Valor Final Estimado: R$ {valor_total - economia_estimada:,.2f}")
    
    print(f"\n⏰ Relatório gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    
    return resultado

# Gerar relatório de exemplo
if models:
    material_exemplo = list(models.keys())[0]
    
    params_relatorio = {
        'fornecedor': 'Alpha Ltda',
        'quantidade': 300,
        'prazo_entrega': 12,
        'condicoes_pagamento': 'À vista',
        'frete_incluso': 'Não',
        'validade_proposta': 45,
        'preco_unitario': 375.00
    }
    
    relatorio = generate_prediction_report(material_exemplo, params_relatorio)

## 8. Exportar Resultados

In [ ]:
# Criar diretório de resultados
os.makedirs("../predicts", exist_ok=True)

# Salvar exemplo de previsão
if models and 'relatorio' in locals():
    # Salvar resultado principal
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    output_file = f"../predicts/previsao_{material_exemplo}_{timestamp}.json"
    with open(output_file, 'w') as f:
        json.dump(relatorio, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Previsão salva em: {output_file}")
    
    # Criar resumo da sessão
    session_summary = {
        'timestamp': datetime.now().isoformat(),
        'modelos_carregados': len(models),
        'materiais_disponiveis': list(models.keys()),
        'exemplo_previsao': relatorio,
        'status': 'Inferência concluída com sucesso'
    }
    
    summary_file = f"../predicts/sessao_inferencia_{timestamp}.json"
    with open(summary_file, 'w') as f:
        json.dump(session_summary, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Resumo da sessão salvo em: {summary_file}")

print(f"\n🎯 Próximos passos:")
print(f"   1️⃣ Execute o notebook 04_sagemaker_deployment.ipynb para deploy na AWS")
print(f"   2️⃣ Use as funções deste notebook para novas previsões")
print(f"   3️⃣ Integre com sua aplicação usando os arquivos salvos")
print(f"\n💡 Dica: Você pode usar a função predict_discount() diretamente para fazer novas previsões!")